# 🔐 pypdpg — your numpy pipeline, now on encrypted data

**Org A** owns sensitive data. **The Vendor** owns a scoring model built on plain numpy.
Today, Org A must hand over plaintext for the model to run. With `pypdpg`, Org A ships
**ciphertext** instead — and the vendor's numpy code runs on it **unchanged**.

Encrypted in → encrypted out → only Org A can decrypt. Homomorphic encryption
(CKKS) behind a numpy interface, by [PDPG-lab](https://pdpglab.xyz).

*This notebook runs top-to-bottom on a fresh Colab runtime in under two minutes.*

In [ ]:
%%time
%pip install -q git+https://github.com/PDPG-lab/pypdpg

---
# 🏦 ACT 1 — Org A: encrypt and ship

Org A holds 200 loan applications with five sensitive features. Under GDPR they cannot
hand this to a third-country vendor in the clear. So they don't.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
N = 200

income      = rng.normal(58_000, 18_000, N).clip(18_000, 150_000)
debt        = rng.normal(22_000, 12_000, N).clip(0, 90_000)
age         = rng.uniform(21, 70, N)
tenure      = rng.uniform(0, 25, N)
utilization = rng.beta(2, 5, N)

X_plain = np.column_stack([income, debt, age, tenure, utilization])
FEATURES = ["income", "debt", "age", "tenure", "utilization"]

print(f"{N} applicants x {len(FEATURES)} features")
print("      " + "".join(f"{f:>13}" for f in FEATURES))
for i in range(3):
    print(f"  [{i}]" + "".join(f"{v:13,.1f}" for v in X_plain[i]))
print("  ...")

In [ ]:
import pypdpg as pdpg

ctx = pdpg.Context.create()          # CKKS: degree 16384, depth 4, ~128-bit security
ctx.save("orga.key")                 # full context incl. secret key — NEVER leaves Org A
ctx.save_public("vendor.ctx")        # public + evaluation keys only — safe to ship

pdpg.encrypt(X_plain, ctx).save("data.enc")
print("shipped: data.enc + vendor.ctx   (kept home: orga.key)")

In [ ]:
import os

plain_bytes = X_plain.nbytes
enc_bytes = os.path.getsize("data.enc")
print(f"plaintext:  {plain_bytes / 1e3:8.1f} kB")
print(f"ciphertext: {enc_bytes / 1e6:8.1f} MB   (~{enc_bytes / plain_bytes:.0f}x — the price of privacy today)")
print(f"vendor.ctx: {os.path.getsize('vendor.ctx') / 1e6:8.1f} MB   (one-time: the vendor's evaluation keys)")

---
# 🏢 ACT 2 — The Vendor: score blind

Everything below the line runs with **no secret key in the process**. The vendor has two
files: `vendor.ctx` (evaluation keys) and `data.enc` (ciphertext). Two lines of setup,
then their existing pipeline.

In [ ]:
import pypdpg as pdpg

pdpg.activate("vendor.ctx")   # line 1: load the evaluation context
pdpg.install()                # line 2: teach np.load about .enc files

### The vendor's existing model code — untouched

In [ ]:
def score(X, w, b):
    """The vendor's credit scorer. Written years ago. Knows nothing about encryption."""
    return X @ w + b

w = np.array([0.001, -0.002, 0.8, 3.0, -150.0])
b = 600.0

# sanity check on a plain sample applicant — ordinary floats in, ordinary floats out
sample = np.array([[60_000, 20_000, 35, 5.0, 0.3]])
print("plain sample score:", score(sample, w, b))

In [ ]:
X = np.load("data.enc")   # same call the vendor always makes
X

In [ ]:
%%time
scores = score(X, w, b)   # the same function, now computing blind
scores

In [ ]:
# encrypted analytics work too — portfolio means, computed without seeing a single value
means = X.mean(axis=0)
means.save("means.enc")
means

In [ ]:
# the closest thing to a peek the vendor gets:
print(X[:, 0])        # column select is allowed — but it's still ciphertext
scores.save("result.enc")

---
# 🏦 ACT 3 — Org A: decrypt the verdict

`result.enc` comes home. Only `orga.key` can open it.

In [ ]:
pdpg.activate("orga.key")

result = pdpg.load("result.enc").decrypt()
true_scores = score(X_plain, w, b)          # plaintext ground truth, for honesty

print("first five scores:", np.round(result[:5], 2))
err = np.abs(result - true_scores).max()
print(f"max abs error vs plaintext: {err:.2e}")
print("CKKS is approximate — off by ~1e-4 on ~600-point scores. Fine for scoring, and we say so.")
assert np.allclose(result, true_scores, atol=1e-2)

portfolio_means = pdpg.load("means.enc").decrypt()
print("decrypted portfolio means:", np.round(portfolio_means, 2))
print("plaintext portfolio means:", np.round(X_plain.mean(axis=0), 2))

---
# 🧨 ACT 4 — FAFO

Try to break it. Every operation that would *reveal* something answers with a teaching
error, not a stack trace. (`fafo` just runs the attempt and prints the refusal, so the
notebook can run top-to-bottom.)

In [ ]:
from pypdpg import EncryptedOperationError

pdpg.activate("vendor.ctx")   # back in the vendor's seat — no secret key here
X = np.load("data.enc")

def fafo(attempt):
    try:
        attempt()
        print("!? that unexpectedly worked")
    except EncryptedOperationError as e:
        print(f"⛔ {e}")

In [ ]:
fafo(lambda: X > 600)          # who passed? not your call to make

In [ ]:
fafo(lambda: np.exp(X))        # transcendental functions don't exist here

In [ ]:
fafo(lambda: X / X)            # no ciphertext division

In [ ]:
fafo(lambda: np.sort(X))       # sorting means comparing means reading

In [ ]:
fafo(lambda: bool(X))          # not even a yes/no leaks

In [ ]:
fafo(lambda: np.asarray(X))    # numpy can't materialize the plaintext either

In [ ]:
# 🫵 your turn — try anything (raw, no safety net; errors here are the point)
# X ** 0.5, X[0], abs(X), np.argmax(X), w @ X, X.sum() ...

---
# What drops in, what needs a rewrite, what's coming

| | |
|---|---|
| ✅ **drop in now** | existing numpy code, unchanged: `+ - * /scalar` · `@` · `dot` · `sum` · `mean` · `square` · `**n` · `pdpg.approx.sigmoid` · column select · save/load · `np.load` |
| 🔁 **needs a rewrite** | data-dependent logic, written branchless (constant-time style), runs today: `if/else` → `gate*b + (1-gate)*c` · thresholds → `sigmoid` gates · filtering → full-shape masking · `/ cipher` → multiply by reciprocal |
| 🔜 **waiting on the engine** | exact comparisons, `max`/`sort` (programmable bootstrapping) · ciphertext division · `exp`/`log`/`sqrt` · unlimited depth · encrypted@encrypted matmul · GPU. They'll drop in — your code won't change |

One thing fits no bucket, ever: *this* party **reading** the data. Not a
roadmap item — the security guarantee.

<sub>A [PDPG-lab](https://pdpglab.xyz) project. Current backend:
[TenSEAL](https://github.com/OpenMined/TenSEAL) (CKKS).</sub>